# MCP Data Analyst - Tier 3 Test Notebook
## Interactive test of advanced tools: reports, charts, dashboards

This notebook tests every Tier 3 tool by calling `engine.py` directly.
Some tools require heavy dependencies (ydata-profiling, sweetviz, autoviz,
streamlit) and may be skipped if not installed.

In [ ]:
import sys
import shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from servers.data_advanced.engine import (
    generate_profile_report,
    generate_sweetviz_report,
    generate_autoviz_report,
    generate_chart,
    generate_dashboard,
)
import pandas as pd

## 00 - Setup test fixtures

In [ ]:
WORK = Path("test_workspace_t3")
WORK.mkdir(exist_ok=True)

RICH_CSV = WORK / "rich_data.csv"
RICH_CSV.write_text("""Region,Product,Revenue,Units_Sold,Order_Date,Discount,Customer_Score
West,Widget A,5000,10,2024-01-15,0.1,85
West,Widget B,3200,8,2024-02-20,0.12,72
East,Widget A,7500,15,2024-03-10,0.05,91
South,Widget C,2100,5,2024-04-05,0.2,60
North,Widget A,4800,12,2024-05-12,0.08,78
West,Widget A,6000,12,2024-06-18,0.07,88
East,Widget B,3000,7,2024-07-22,0.15,65
South,Widget A,2500,6,2024-08-30,0.1,70
North,Widget C,1800,4,2024-09-14,0.25,55
West,Widget C,4200,9,2024-10-01,0.09,80
East,Widget A,8000,16,2024-11-05,0.03,95
South,Widget B,1500,3,2024-12-10,0.3,45
North,Widget A,5500,14,2024-01-25,0.06,82
West,Widget B,2800,7,2024-02-14,0.11,68
East,Widget C,3500,8,2024-03-28,0.13,75
""")

GEO_JSON = WORK / "states.geojson"
GEO_JSON.write_text('{"type":"FeatureCollection","features":['
    '{"type":"Feature","properties":{"name":"California","code":"CA"},'
    '"geometry":{"type":"Polygon","coordinates":[[[-120,35],[-120,40],[-115,40],[-115,35],[-120,35]]]}},'
    '{"type":"Feature","properties":{"name":"Texas","code":"TX"},'
    '"geometry":{"type":"Polygon","coordinates":[[[-105,26],[-105,36],[-95,36],[-95,26],[-105,26]]]}},'
    '{"type":"Feature","properties":{"name":"New York","code":"NY"},'
    '"geometry":{"type":"Polygon","coordinates":[[[-80,40],[-80,45],[-72,45],[-72,40],[-80,40]]]}}'
    ']}')

print(f"Workspace: {WORK.absolute()}")
print("Files created: rich_data.csv, states.geojson")

## 01 - generate_profile_report (ydata-profiling)

In [ ]:
r = generate_profile_report(str(RICH_CSV), minimal=True, correlations=False)
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    assert r["success"] is True
    print(f"Report: {r['report_path']}")
    print(f"Size: {r['report_size_kb']:,} KB")
    print(f"Columns profiled: {r['columns_profiled']}")
    print(f"Rows: {r['rows']}")
    assert Path(WORK / r["report_path"]).exists()
    print("PASS: generate_profile_report - minimal")

In [ ]:
r = generate_profile_report(str(RICH_CSV), minimal=False, correlations=True,
                            title="Full Profile", description="Test report")
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    print(f"Report: {r['report_path']}")
    print(f"Size: {r['report_size_kb']:,} KB")
    print(f"Correlations included: {r['correlations_included']}")
    print("PASS: generate_profile_report - full")

In [ ]:
r = generate_profile_report(str(WORK / "nonexistent.csv"))
assert r["success"] is False
print(f"Error: {r['error']}")
print("PASS: generate_profile_report - file not found")

## 02 - generate_sweetviz_report

In [ ]:
r = generate_sweetviz_report(str(RICH_CSV))
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    assert r["success"] is True
    print(f"Report: {r['report_path']}")
    print(f"Size: {r['report_size_kb']:,} KB")
    print(f"Columns analysed: {r['columns_analysed']}")
    assert Path(WORK / r["report_path"]).exists()
    print("PASS: generate_sweetviz_report")

In [ ]:
r = generate_sweetviz_report(str(RICH_CSV), target_column="Revenue")
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    print(f"Target: {r['target_column']}")
    print("PASS: generate_sweetviz_report - with target")

In [ ]:
r = generate_sweetviz_report(str(RICH_CSV), target_column="NonExistent")
assert r["success"] is False
print(f"Error: {r['error']}")
print("PASS: generate_sweetviz_report - bad target")

## 03 - generate_autoviz_report

In [ ]:
r = generate_autoviz_report(str(RICH_CSV), chart_format="html", max_rows_analyzed=100)
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    assert r["success"] is True
    print(f"Output dir: {r['output_dir']}")
    print(f"Chart count: {r['chart_count']}")
    print(f"Rows analyzed: {r['rows_analyzed']}")
    print(f"Files: {r['chart_files'][:5]}...")
    print("PASS: generate_autoviz_report")

## 04 - generate_chart

In [ ]:
r = generate_chart(str(RICH_CSV), chart_type="bar", value_column="Revenue",
                   category_column="Region", agg_func="sum")
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    assert r["success"] is True
    print(f"Chart: {r['output_path']}")
    print(f"Title: {r['title']}")
    print(f"Rows plotted: {r['rows_plotted']}")
    assert Path(WORK / r["output_path"]).exists()
    print("PASS: generate_chart - bar")

In [ ]:
r = generate_chart(str(RICH_CSV), chart_type="pie", value_column="Revenue",
                   category_column="Product", agg_func="sum")
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    print(f"Chart: {r['output_path']}")
    print("PASS: generate_chart - pie")

In [ ]:
r = generate_chart(str(RICH_CSV), chart_type="time_series", value_column="Revenue",
                   date_column="Order_Date", period="M")
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    print(f"Chart: {r['output_path']}")
    print("PASS: generate_chart - time_series")

In [ ]:
r = generate_chart(str(RICH_CSV), chart_type="treemap", value_column="Revenue",
                   hierarchy_columns=["Region", "Product"])
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    print(f"Chart: {r['output_path']}")
    print("PASS: generate_chart - treemap")

In [ ]:
r = generate_chart(str(RICH_CSV), chart_type="scatter", value_column="Revenue",
                   category_column="Units_Sold")
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    print(f"Chart: {r['output_path']}")
    print("PASS: generate_chart - scatter")

In [ ]:
r = generate_chart(str(RICH_CSV), chart_type="heatmap", value_column="Revenue")
assert r["success"] is False
print(f"Error: {r['error']}")
print(f"Hint: {r['hint']}")
print("PASS: generate_chart - invalid type")

## 05 - generate_dashboard

In [ ]:
r = generate_dashboard(str(RICH_CSV), dry_run=True)
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    assert r["dry_run"] is True
    print(f"Would generate: {r['would_generate']}")
    print("PASS: generate_dashboard - dry_run")

In [ ]:
r = generate_dashboard(str(RICH_CSV), title="Sales Dashboard",
                       chart_types=["bar", "pie", "scatter"])
if not r["success"]:
    print(f"SKIPPED: {r['error']}")
else:
    assert r["success"] is True
    print(f"Output: {r['output_path']}")
    print(f"Title: {r['dashboard_title']}")
    print(f"Charts: {r['charts_included']}")
    print(f"KPI columns: {r['kpi_columns']}")
    print(f"Filter columns: {r['filter_columns']}")
    print(f"Run command: {r['run_command']}")
    assert Path(WORK / r["output_path"]).exists()
    print("PASS: generate_dashboard")

In [ ]:
app_path = WORK / "app.py"
if app_path.exists():
    import py_compile
    try:
        py_compile.compile(str(app_path), doraise=True)
        print("PASS: generate_dashboard - generated code is valid Python")
    except py_compile.PyCompileError as e:
        print(f"FAIL: Generated code has syntax error: {e}")

## 06 - File not found errors

In [ ]:
for tool_name, tool_fn in [
    ("generate_profile_report", generate_profile_report),
    ("generate_sweetviz_report", generate_sweetviz_report),
    ("generate_autoviz_report", generate_autoviz_report),
    ("generate_chart", lambda: generate_chart(str(WORK / "no.csv"), "bar", "Revenue")),
    ("generate_dashboard", generate_dashboard),
]:
    if tool_name == "generate_chart":
        r = tool_fn()
    else:
        r = tool_fn(str(WORK / "nonexistent.csv"))
    assert r["success"] is False, f"{tool_name} should fail for missing file"
    assert "hint" in r
    print(f"  {tool_name}: error handled correctly")

print("PASS: all tools handle missing files")

## 07 - Cleanup

In [ ]:
# Uncomment to clean up test workspace
# shutil.rmtree(WORK)
# print("Workspace cleaned")

print(f"\nWorkspace preserved at: {WORK.absolute()}")
print("All Tier 3 tools tested successfully.")